## Pipeline 4B — Social Media Timing Optimizer (Predictive)

This notebook trains a predictive model to estimate `engagement_rate` from **timing + format** features only, then saves artifacts and (optionally) runs inference to pre-compute a full platform × day × hour recommendation matrix.

### Files this notebook uses
- `ml/social_media_timing/features.py`
- `ml/social_media_timing/etl.py`
- `ml/social_media_timing/artifacts.py`
- `ml/social_media_timing/infer.py`
- `ml/utils_db.py`, `ml/config.py`

### Target
- `engagement_rate` (continuous)

### Notes
- This pipeline intentionally **excludes content features** (sentiment/topic/etc.). Use Pipeline 4 for content guidance.


## 1) Problem framing

**Business question:** Given the platform, day of week, and time of day, what engagement rate can we expect — and what combination maximizes reach?

**Why engagement_rate (not donation_referrals):** Engagement rate is consistently measured by every platform, so it’s always available even when donation tracking/UTMs vary.

**Separation from Pipeline 4:**
- Pipeline 4 focuses on **what to post** (content features) and interpretability.
- Pipeline 4B focuses on **when/where to post** (timing/format features) and prediction accuracy.


## Setup — install required packages

Run this once per environment/kernel. It installs the repo’s ML dependencies from `ml/requirements.txt`.


In [1]:
# Ensure the repo root is on sys.path so `import ml...` works
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()

# Walk upward until we find the repo root containing the `ml/` package directory
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "ml" / "__init__.py").exists():
        repo_root = p
        break

if repo_root is None:
    raise RuntimeError(f"Could not find repo root containing ml/__init__.py starting from {cwd}")

sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

print("Repo root:", repo_root)
print("Python:", sys.executable)
print("CWD:", Path.cwd())


Repo root: /Users/brytongustin/Downloads/Winter_Junior_Core/INTEX II/intex2
Python: /usr/local/bin/python3.12
CWD: /Users/brytongustin/Downloads/Winter_Junior_Core/INTEX II/intex2


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    ExtraTreesRegressor,
    StackingRegressor,
)

from ml.social_media_timing.etl import build_training_frame
from ml.social_media_timing.artifacts import save_model_bundle, save_metadata, save_metrics

# Optional: run DB-writing inference at the end
from ml.social_media_timing.infer import run_inference

print("Imports OK")


Imports OK


## 2) Data acquisition & preparation

This uses the ETL wrapper to fetch `social_media_posts` from Supabase and build the modeling frame.

- X features: platform, post_hour, day_of_week, media_type, is_boosted, boost_budget_php, has_call_to_action, post_type, is_weekend
- y target: engagement_rate


In [3]:
X, y = build_training_frame()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("y mean:", float(np.mean(y)))

# Basic sanity checks
assert len(X) == len(y)
assert np.isfinite(y).all()

X.head()


X shape: (812, 30)
y shape: (812,)
y mean: 0.09898017241379312


,is_boosted,has_call_to_action,is_weekend,post_hour,boost_budget_php,platform_Facebook,platform_Instagram,platform_LinkedIn,platform_TikTok,platform_Twitter,...,media_type_Photo,media_type_Reel,media_type_Text,media_type_Video,post_type_Campaign,post_type_EducationalContent,post_type_EventPromotion,post_type_FundraisingAppeal,post_type_ImpactStory,post_type_ThankYou
0,0,1,0,18,0.00,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
1,0,0,0,11,0.00,False,True,False,False,False,...,True,False,False,False,False,True,False,False,False,False
2,0,0,1,10,0.00,False,False,True,False,False,...,False,False,True,False,False,False,True,False,False,False
3,0,0,0,15,0.00,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,True
4,1,1,0,15,4030.64,False,False,False,True,False,...,False,True,False,False,False,False,False,False,False,True


## 3) Exploration

The pipeline spec documents these confirmed findings (from the dataset):
- `post_hour` vs `engagement_rate`: r = 0.444 (dominant signal)
- Best days by engagement: Saturday (0.1057), Friday (0.1046), Sunday (0.1029)

The cells below compute a few quick checks so you can validate your environment + data.


In [4]:
# Correlation proxy for post_hour using the raw (non-one-hot) column if present
# Here post_hour exists as numeric in X by design.
if "post_hour" in X.columns:
    r = np.corrcoef(X["post_hour"].astype(float).values, y.astype(float).values)[0, 1]
    print("corr(post_hour, engagement_rate) =", float(r))

# Weekend vs weekday mean
if "is_weekend" in X.columns:
    weekend_mean = float(y[X["is_weekend"].astype(int).eq(1)].mean())
    weekday_mean = float(y[X["is_weekend"].astype(int).eq(0)].mean())
    print("mean engagement (weekend):", weekend_mean)
    print("mean engagement (weekday):", weekday_mean)


corr(post_hour, engagement_rate) = 0.4437435019119868
mean engagement (weekend): 0.10437935779816514
mean engagement (weekday): 0.0969986531986532


## 4) Train/test split

We’ll use a holdout test set for final metrics and 5-fold CV on the training split for model comparison.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("train:", X_train.shape, "test:", X_test.shape)

cv = KFold(n_splits=5, shuffle=True, random_state=42)


train: (649, 30) test: (163, 30)


## 5) Modeling — compare models (5-fold CV)

Per spec, we train and compare:
- Linear Regression (baseline)
- Decision Tree Regressor
- KNN Regressor
- SVR
- Random Forest Regressor
- Gradient Boosting Regressor
- AdaBoost Regressor
- Extra Trees Regressor
- Stacking Regressor

This section computes a head-to-head comparison table with CV MAE and CV R².


In [6]:
def evaluate_models(models: dict[str, object]) -> pd.DataFrame:
    rows: list[dict] = []
    for name, model in models.items():
        scoring = {
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        }
        results = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=None,
            return_train_score=False,
        )
        mae = -results["test_mae"]
        r2 = results["test_r2"]
        rows.append(
            {
                "model": name,
                "cv_mae_mean": float(np.mean(mae)),
                "cv_mae_std": float(np.std(mae)),
                "cv_r2_mean": float(np.mean(r2)),
                "cv_r2_std": float(np.std(r2)),
            }
        )
    return pd.DataFrame(rows).sort_values(["cv_mae_mean"], ascending=True).reset_index(drop=True)


models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "KNN": Pipeline([("scaler", StandardScaler(with_mean=False)), ("model", KNeighborsRegressor())]),
    "SVR": Pipeline([("scaler", StandardScaler(with_mean=False)), ("model", SVR())]),
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=300),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42, n_estimators=500),
}

# Simple stacking baseline (tree + linear) with passthrough
stack_estimators = [
    ("rf", RandomForestRegressor(random_state=42, n_estimators=200)),
    ("gbr", GradientBoostingRegressor(random_state=42)),
]
models["Stacking"] = StackingRegressor(
    estimators=stack_estimators,
    final_estimator=LinearRegression(),
    passthrough=True,
    n_jobs=None,
)

comparison = evaluate_models(models)
comparison


,model,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std
0,GradientBoosting,0.028671,0.002382,0.522198,0.033335
1,Stacking,0.028910,0.002679,0.507103,0.038823
2,RandomForest,0.029053,0.002723,0.502118,0.051301
3,ExtraTrees,0.031212,0.003461,0.419138,0.064100
4,AdaBoost,0.032177,0.001850,0.416340,0.028182
5,DecisionTree,0.037746,0.004963,0.109923,0.156133
6,LinearRegression,0.038970,0.003652,0.199331,0.063494
7,KNN,0.046501,0.004180,-0.095515,0.076927
8,SVR,0.052915,0.003376,-0.280350,0.108207


## 6) Select best model, fit on train, evaluate on test

We’ll pick the best model by **lowest CV MAE** (primary metric), then report test MAE, RMSE, and R².


In [7]:
best_name = str(comparison.iloc[0]["model"])
best_model = models[best_name]

print("Selected:", best_name)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = mean_squared_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("Test MAE:", float(test_mae))
print("Test RMSE:", float(test_rmse))
print("Test R2:", float(test_r2))

mean_engagement = float(np.mean(y))
if mean_engagement > 0:
    print(
        "Business interp: off by", float(test_mae),
        "engagement points on avg; mean engagement is", mean_engagement,
        "(~", float(test_mae / mean_engagement * 100.0), "% of mean)",
    )


Selected: GradientBoosting
Test MAE: 0.03164213801694636
Test RMSE: 0.0017932181944333608
Test R2: 0.4746549212667266
Business interp: off by 0.03164213801694636 engagement points on avg; mean engagement is 0.09898017241379312 (~ 31.96815811217657 % of mean)


## 7) Save artifacts (model + metadata + metrics)

This writes to:
- `models/social-media-timing/model.sav`
- `models/social-media-timing/model.json` (append-only run log with metadata + metrics)

The infer job uses the saved `feature_list` to keep one-hot columns aligned.

In [8]:
feature_list = list(X.columns)

# Save bundle
save_model_bundle(best_model, feature_list=feature_list)

# Save metadata/metrics
_ = save_metadata(
    feature_list=feature_list,
    model_type=best_name,
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(X),
)
_ = save_metrics(
    mae=float(test_mae),
    rmse=float(test_rmse),
    r2=float(test_r2),
    cv_table=comparison.to_dict(orient="records"),
)

print("Saved model + metadata + metrics")


Saved model + metadata + metrics
